# Figure regeneration — Multi-Representation Fusion for TDC ADMET Prediction

Regenerates the five main-text figures (figure1–figure5.png) from the ground-truth summaries produced by the `results*.ipynb` analyses in this folder.

- **Self-contained**: derived summary tables (`ground_truth`, `pertask`, `stat_tests`) are embedded in the setup cell.
- **External input**: `uniform_val_test_unified.csv` (in this folder) for Figure 4's validation→test scatter.
- **Outputs**: `figure1.png` … `figure5.png` written next to this notebook.

Run all cells top to bottom. Requires: numpy, pandas, matplotlib, scipy.

In [ ]:
# === Setup: figure/table regeneration for the ADMET multi-representation fusion paper ===
# Self-contained. Reads only files that live in this organize/ folder.
# Derived summary tables (ground_truth, pertask) are embedded below so the notebook
# runs standalone; the raw val->test grid is read from uniform_val_test_unified.csv.
import os, json, base64, io
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import spearmanr, kendalltau, wilcoxon

HERE = os.path.dirname(os.path.abspath("__file__")) if "__file__" in globals() else os.getcwd()
CSV_PATH = os.path.join(HERE, "uniform_val_test_unified.csv")
OUTDIR = HERE

META_GREY = "#888888"

def apply_figure_style(*, frame="open", font=None, sizes=(8, 7, 6), grid=False):
    import matplotlib as mpl
    if frame not in ("open", "boxed", "none"):
        raise ValueError(f"frame must be 'open'|'boxed'|'none', got {frame!r}")
    try:
        import os, sys, glob, matplotlib.font_manager as fm
        fdir = os.path.join(os.environ.get("CONDA_PREFIX") or sys.prefix, "fonts")
        if os.path.isdir(fdir):
            known = {f.fname for f in fm.fontManager.ttflist}
            for f in glob.glob(os.path.join(fdir, "*.ttf")):
                if f not in known:
                    fm.fontManager.addfont(f)
    except Exception:
        pass
    base, secondary, tick = sizes
    boxed = (frame == "boxed")
    rc = {
        "font.family": "sans-serif",
        "font.size": base,
        "axes.labelsize": base,
        "axes.titlesize": base,
        "legend.fontsize": secondary,
        "xtick.labelsize": tick,
        "ytick.labelsize": tick,
        "axes.linewidth": 0.6,
        "xtick.direction": "out", "ytick.direction": "out",
        "xtick.major.size": 3, "ytick.major.size": 3,
        "xtick.major.width": 0.6, "ytick.major.width": 0.6,
        "axes.spines.top": boxed, "axes.spines.right": boxed,
        "axes.spines.left": frame != "none", "axes.spines.bottom": frame != "none",
        "axes.grid": bool(grid),
        "legend.frameon": False,
        "figure.dpi": 200,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "axes.titleweight": "normal",
        "axes.titlelocation": "left",
        "axes.labelweight": "normal",
        "lines.linewidth": 1.2,
        "patch.linewidth": 0.6,
        "pdf.fonttype": 42, "ps.fonttype": 42,
    }
    if font:
        rc["font.sans-serif"] = [font, "DejaVu Sans"]
    mpl.rcParams.update(rc)


# --- embedded derived summaries (ground truth from results*.ipynb) ---
_GT = json.loads(r"""{"per_model_norm": {"stl_chemeleon": [0.6454, 8.19, 0, 3, 6, 16], "mtl_map": [0.6215, 8.67, 2, 3, 5, 14], "mtl_chemeleon": [0.6148, 8.76, 0, 2, 4, 16], "mtl_dmpnn": [0.6121, 8.76, 2, 4, 6, 14], "stl_map": [0.6092, 8.81, 0, 3, 4, 13], "mtl_molformer": [0.5685, 9.76, 2, 3, 4, 13], "mtl_graphmvp": [0.5194, 10.76, 1, 4, 4, 10], "stl_molformer": [0.4653, 11.81, 0, 1, 1, 7], "stl_graphmvp": [0.4261, 12.38, 0, 1, 1, 5], "stl_dmpnn": [0.2224, 16.57, 0, 0, 0, 2]}, "map_agg": {"mtl": {"map": 0.621, "mean": 0.555, "max": 0.673, "map_gt_mean": "12/21", "map_gt_max": "3/21"}, "stl": {"map": 0.609, "mean": 0.518, "max": 0.664, "map_gt_mean": "15/21", "map_gt_max": "3/21"}}, "ksweep": [[1, 0.5352, null, 0.5527, null], [2, 0.7221, 0.1869, 0.73, 0.1773], [3, 0.7648, 0.0427, 0.7623, 0.0323], [4, 0.7825, 0.0177, 0.7764, 0.0141], [5, 0.7877, 0.0052, 0.7816, 0.0052], [6, 0.7903, 0.0026, 0.7819, 0.0003], [7, 0.7878, -0.0025, 0.7785, -0.0034], [8, 0.7804, -0.0074, 0.773, -0.0055], [9, 0.7751, -0.0053, 0.7672, -0.0058], [10, 0.7805, 0.0054, 0.7672, 0.0]], "final_ensemble": ["mtl_dmpnn", "mtl_molformer", "stl_chemeleon", "stl_graphmvp", "stl_map"], "final_composition": {"MTL": ["dmpnn", "molformer"], "STL": ["chemeleon", "graphmvp", "map"]}, "headline": {"22-task": {"within": [0.815, 4.73, 2, 11, 15], "cross": [0.856, 3.91, 6, 14, 19]}, "21-task": {"within": [0.854, 3.95, 2, 11, 15], "cross": [0.897, 3.1, 6, 14, 19]}}, "sota_tasks": {"within": ["Lipophilicity_AstraZeneca", "PPBR_AZ"], "cross": ["HIA_Hou", "Pgp_Broccatelli", "hERG", "Lipophilicity_AstraZeneca", "PPBR_AZ", "Clearance_Microsome_AZ"]}, "loo_shapley": {"mtl_dmpnn": [0.6047, 0.0577, 0.6851, 0.0605, 0.2079, 0.2093], "stl_map": [0.6045, 0.022, 0.7745, 0.0088, 0.1763, 0.2034], "stl_graphmvp": [0.416, -0.0051, 0.5241, -0.013, 0.0882, 0.0989], "mtl_molformer": [0.5541, 0.0592, 0.6439, 0.0468, 0.1852, 0.1889], "stl_chemeleon": [0.6331, 0.0417, 0.7298, 0.0182, 0.1963, 0.1967]}, "full_ensemble_norm": {"within": 0.8539, "cross": 0.8971}, "corr": {"cross": {"pred_offdiag": 0.878, "resid_offdiag": 0.899}, "within": {"pred_offdiag": 0.767, "resid_offdiag": 0.837}}, "winners": {"cross": {"mtl_dmpnn": 6, "mtl_molformer": 6, "stl_map": 6, "stl_chemeleon": 2, "stl_graphmvp": 1}, "within_taskseed": {"mtl_dmpnn": 36, "stl_chemeleon": 27, "stl_map": 21, "mtl_molformer": 20, "stl_graphmvp": 1}}, "best_single": {"cross": {"model": "stl_map", "mean_rank": 5.476, "ens_rank": 3.095}, "within": {"model": "stl_chemeleon", "mean_rank": 8.429, "ens_rank": 3.952}}, "val_test_corr": {"within": {"rho": 0.796, "tau": 0.603, "ci": [0.71, 0.859]}, "cross": {"rho": 0.748, "tau": 0.561, "ci": [0.643, 0.826]}, "combined": {"rho": 0.805, "tau": 0.616, "ci": [0.725, 0.862]}, "n_configs": 133, "coarse": {"within_pct": 99, "within_bestsingle": 0.633, "cross_pct": 91, "cross_bestsingle": 0.786}}, "k4_variant": {"drop": "stl_graphmvp", "cross_norm": 0.9101, "cross_rank": 2.81, "full_cross_norm": 0.8971, "full_cross_rank": 3.1}, "k5_selection": {"simple_combo": ["mtl_dmpnn", "mtl_molformer", "stl_chemeleon", "stl_graphmvp", "stl_map"], "val_within": 0.7923, "val_cross": 0.7883, "combined": 1.5806, "n_candidates_meta": 3069, "n_subsets": 1023, "k_candidates": 161}}""")
_PT = json.loads(r"""{"t1": [{"group": "oral_uptake", "dataset": "Caco2_Wang", "n": "910", "metric": "mae", "mtl_dmpnn": {"raw": 0.322, "norm": 0.62}, "mtl_chemeleon": {"raw": 0.359, "norm": 0.38}, "mtl_graphmvp": {"raw": 0.393, "norm": 0.38}, "mtl_molformer": {"raw": 0.401, "norm": 0.29}, "stl_dmpnn": {"raw": 0.463, "norm": 0.25}, "stl_chemeleon": {"raw": 0.462, "norm": 0.25}, "stl_graphmvp": {"raw": 0.432, "norm": 0.29}, "stl_molformer": {"raw": 0.42, "norm": 0.29}}, {"group": "oral_uptake", "dataset": "HIA_Hou", "n": "578", "metric": "auroc", "mtl_dmpnn": {"raw": 0.991, "norm": 0.95}, "mtl_chemeleon": {"raw": 0.988, "norm": 0.8}, "mtl_graphmvp": {"raw": 0.929, "norm": 0.2}, "mtl_molformer": {"raw": 0.981, "norm": 0.6}, "stl_dmpnn": {"raw": 0.851, "norm": 0.1}, "stl_chemeleon": {"raw": 0.958, "norm": 0.35}, "stl_graphmvp": {"raw": 0.953, "norm": 0.35}, "stl_molformer": {"raw": 0.977, "norm": 0.55}}, {"group": "oral_uptake", "dataset": "Solubility_AqSolDB", "n": "9982", "metric": "mae", "mtl_dmpnn": {"raw": 0.822, "norm": 0.61}, "mtl_chemeleon": {"raw": 0.95, "norm": 0.28}, "mtl_graphmvp": {"raw": 0.983, "norm": 0.28}, "mtl_molformer": {"raw": 0.916, "norm": 0.39}, "stl_dmpnn": {"raw": 1.102, "norm": 0.06}, "stl_chemeleon": {"raw": 0.828, "norm": 0.5}, "stl_graphmvp": {"raw": 0.897, "norm": 0.44}, "stl_molformer": {"raw": 0.868, "norm": 0.44}}, {"group": "distribution_cns", "dataset": "BBB_Martins", "n": "2030", "metric": "auroc", "mtl_dmpnn": {"raw": 0.897, "norm": 0.54}, "mtl_chemeleon": {"raw": 0.909, "norm": 0.73}, "mtl_graphmvp": {"raw": 0.889, "norm": 0.38}, "mtl_molformer": {"raw": 0.903, "norm": 0.62}, "stl_dmpnn": {"raw": 0.722, "norm": 0.0}, "stl_chemeleon": {"raw": 0.892, "norm": 0.5}, "stl_graphmvp": {"raw": 0.879, "norm": 0.38}, "stl_molformer": {"raw": 0.864, "norm": 0.31}}, {"group": "distribution_cns", "dataset": "Lipophilicity_AstraZeneca", "n": "4200", "metric": "mae", "mtl_dmpnn": {"raw": 0.516, "norm": 0.71}, "mtl_chemeleon": {"raw": 0.58, "norm": 0.29}, "mtl_graphmvp": {"raw": 0.603, "norm": 0.29}, "mtl_molformer": {"raw": 0.639, "norm": 0.14}, "stl_dmpnn": {"raw": 0.592, "norm": 0.29}, "stl_chemeleon": {"raw": 0.471, "norm": 0.86}, "stl_graphmvp": {"raw": 0.47, "norm": 0.9}, "stl_molformer": {"raw": 0.564, "norm": 0.38}}, {"group": "distribution_cns", "dataset": "PPBR_AZ", "n": "2790", "metric": "mae", "mtl_dmpnn": {"raw": 7.112, "norm": 1.0}, "mtl_chemeleon": {"raw": 7.544, "norm": 0.9}, "mtl_graphmvp": {"raw": 7.535, "norm": 0.9}, "mtl_molformer": {"raw": 8.054, "norm": 0.65}, "stl_dmpnn": {"raw": 10.253, "norm": 0.1}, "stl_chemeleon": {"raw": 7.834, "norm": 0.75}, "stl_graphmvp": {"raw": 7.957, "norm": 0.7}, "stl_molformer": {"raw": 8.241, "norm": 0.6}}, {"group": "distribution_cns", "dataset": "VDss_Lombardo", "n": "1130", "metric": "spearman", "mtl_dmpnn": {"raw": 0.624, "norm": 0.79}, "mtl_chemeleon": {"raw": 0.606, "norm": 0.74}, "mtl_graphmvp": {"raw": 0.582, "norm": 0.68}, "mtl_molformer": {"raw": 0.574, "norm": 0.68}, "stl_dmpnn": {"raw": 0.585, "norm": 0.74}, "stl_chemeleon": {"raw": 0.669, "norm": 0.89}, "stl_graphmvp": {"raw": 0.605, "norm": 0.74}, "stl_molformer": {"raw": 0.605, "norm": 0.74}}, {"group": "interaction_liabilities", "dataset": "CYP2C9_Veith", "n": "12092", "metric": "auprc", "mtl_dmpnn": {"raw": 0.741, "norm": 0.3}, "mtl_chemeleon": {"raw": 0.769, "norm": 0.6}, "mtl_graphmvp": {"raw": 0.741, "norm": 0.3}, "mtl_molformer": {"raw": 0.756, "norm": 0.5}, "stl_dmpnn": {"raw": 0.717, "norm": 0.2}, "stl_chemeleon": {"raw": 0.766, "norm": 0.55}, "stl_graphmvp": {"raw": 0.746, "norm": 0.35}, "stl_molformer": {"raw": 0.756, "norm": 0.5}}, {"group": "interaction_liabilities", "dataset": "CYP2D6_Veith", "n": "13130", "metric": "auprc", "mtl_dmpnn": {"raw": 0.67, "norm": 0.58}, "mtl_chemeleon": {"raw": 0.681, "norm": 0.63}, "mtl_graphmvp": {"raw": 0.647, "norm": 0.47}, "mtl_molformer": {"raw": 0.65, "norm": 0.53}, "stl_dmpnn": {"raw": 0.593, "norm": 0.21}, "stl_chemeleon": {"raw": 0.64, "norm": 0.37}, "stl_graphmvp": {"raw": 0.638, "norm": 0.37}, "stl_molformer": {"raw": 0.642, "norm": 0.37}}, {"group": "interaction_liabilities", "dataset": "CYP3A4_Veith", "n": "12328", "metric": "auprc", "mtl_dmpnn": {"raw": 0.827, "norm": 0.21}, "mtl_chemeleon": {"raw": 0.868, "norm": 0.58}, "mtl_graphmvp": {"raw": 0.833, "norm": 0.26}, "mtl_molformer": {"raw": 0.83, "norm": 0.26}, "stl_dmpnn": {"raw": 0.823, "norm": 0.16}, "stl_chemeleon": {"raw": 0.87, "norm": 0.58}, "stl_graphmvp": {"raw": 0.854, "norm": 0.42}, "stl_molformer": {"raw": 0.834, "norm": 0.26}}, {"group": "interaction_liabilities", "dataset": "Pgp_Broccatelli", "n": "1218", "metric": "auroc", "mtl_dmpnn": {"raw": 0.909, "norm": 0.56}, "mtl_chemeleon": {"raw": 0.923, "norm": 0.62}, "mtl_graphmvp": {"raw": 0.92, "norm": 0.62}, "mtl_molformer": {"raw": 0.908, "norm": 0.5}, "stl_dmpnn": {"raw": 0.85, "norm": 0.12}, "stl_chemeleon": {"raw": 0.937, "norm": 0.94}, "stl_graphmvp": {"raw": 0.88, "norm": 0.25}, "stl_molformer": {"raw": 0.902, "norm": 0.5}}, {"group": "interaction_liabilities", "dataset": "hERG", "n": "655", "metric": "auroc", "mtl_dmpnn": {"raw": 0.846, "norm": 0.7}, "mtl_chemeleon": {"raw": 0.844, "norm": 0.7}, "mtl_graphmvp": {"raw": 0.802, "norm": 0.5}, "mtl_molformer": {"raw": 0.883, "norm": 1.0}, "stl_dmpnn": {"raw": 0.769, "norm": 0.45}, "stl_chemeleon": {"raw": 0.865, "norm": 0.8}, "stl_graphmvp": {"raw": 0.77, "norm": 0.45}, "stl_molformer": {"raw": 0.802, "norm": 0.5}}, {"group": "metabolic_turnover", "dataset": "CYP2C9_Substrate_CarbonMangels", "n": "669", "metric": "auprc", "mtl_dmpnn": {"raw": 0.477, "norm": 1.0}, "mtl_chemeleon": {"raw": 0.429, "norm": 0.75}, "mtl_graphmvp": {"raw": 0.457, "norm": 0.95}, "mtl_molformer": {"raw": 0.419, "norm": 0.75}, "stl_dmpnn": {"raw": 0.355, "norm": 0.15}, "stl_chemeleon": {"raw": 0.427, "norm": 0.75}, "stl_graphmvp": {"raw": 0.377, "norm": 0.35}, "stl_molformer": {"raw": 0.392, "norm": 0.5}}, {"group": "metabolic_turnover", "dataset": "CYP2D6_Substrate_CarbonMangels", "n": "667", "metric": "auprc", "mtl_dmpnn": {"raw": 0.651, "norm": 0.39}, "mtl_chemeleon": {"raw": 0.66, "norm": 0.39}, "mtl_graphmvp": {"raw": 0.684, "norm": 0.5}, "mtl_molformer": {"raw": 0.705, "norm": 0.78}, "stl_dmpnn": {"raw": 0.549, "norm": 0.17}, "stl_chemeleon": {"raw": 0.685, "norm": 0.5}, "stl_graphmvp": {"raw": 0.566, "norm": 0.17}, "stl_molformer": {"raw": 0.655, "norm": 0.39}}, {"group": "metabolic_turnover", "dataset": "CYP3A4_Substrate_CarbonMangels", "n": "670", "metric": "auroc", "mtl_dmpnn": {"raw": 0.651, "norm": 0.79}, "mtl_chemeleon": {"raw": 0.666, "norm": 0.95}, "mtl_graphmvp": {"raw": 0.689, "norm": 1.0}, "mtl_molformer": {"raw": 0.664, "norm": 0.95}, "stl_dmpnn": {"raw": 0.579, "norm": 0.11}, "stl_chemeleon": {"raw": 0.634, "norm": 0.58}, "stl_graphmvp": {"raw": 0.604, "norm": 0.26}, "stl_molformer": {"raw": 0.599, "norm": 0.26}}, {"group": "metabolic_turnover", "dataset": "Clearance_Hepatocyte_AZ", "n": "1213", "metric": "spearman", "mtl_dmpnn": {"raw": 0.426, "norm": 0.44}, "mtl_chemeleon": {"raw": 0.448, "norm": 0.83}, "mtl_graphmvp": {"raw": 0.485, "norm": 0.89}, "mtl_molformer": {"raw": 0.556, "norm": 1.0}, "stl_dmpnn": {"raw": 0.382, "norm": 0.28}, "stl_chemeleon": {"raw": 0.464, "norm": 0.83}, "stl_graphmvp": {"raw": 0.417, "norm": 0.39}, "stl_molformer": {"raw": 0.517, "norm": 0.94}}, {"group": "metabolic_turnover", "dataset": "Clearance_Microsome_AZ", "n": "1102", "metric": "spearman", "mtl_dmpnn": {"raw": 0.595, "norm": 0.65}, "mtl_chemeleon": {"raw": 0.616, "norm": 0.75}, "mtl_graphmvp": {"raw": 0.608, "norm": 0.75}, "mtl_molformer": {"raw": 0.607, "norm": 0.75}, "stl_dmpnn": {"raw": 0.5, "norm": 0.15}, "stl_chemeleon": {"raw": 0.605, "norm": 0.75}, "stl_graphmvp": {"raw": 0.547, "norm": 0.3}, "stl_molformer": {"raw": 0.562, "norm": 0.45}}, {"group": "distal_safety", "dataset": "AMES", "n": "7278", "metric": "auroc", "mtl_dmpnn": {"raw": 0.821, "norm": 0.3}, "mtl_chemeleon": {"raw": 0.848, "norm": 0.65}, "mtl_graphmvp": {"raw": 0.82, "norm": 0.3}, "mtl_molformer": {"raw": 0.823, "norm": 0.3}, "stl_dmpnn": {"raw": 0.812, "norm": 0.2}, "stl_chemeleon": {"raw": 0.84, "norm": 0.45}, "stl_graphmvp": {"raw": 0.802, "norm": 0.2}, "stl_molformer": {"raw": 0.821, "norm": 0.3}}, {"group": "distal_safety", "dataset": "DILI", "n": "475", "metric": "auroc", "mtl_dmpnn": {"raw": 0.929, "norm": 0.95}, "mtl_chemeleon": {"raw": 0.904, "norm": 0.65}, "mtl_graphmvp": {"raw": 0.905, "norm": 0.65}, "mtl_molformer": {"raw": 0.896, "norm": 0.6}, "stl_dmpnn": {"raw": 0.878, "norm": 0.4}, "stl_chemeleon": {"raw": 0.902, "norm": 0.65}, "stl_graphmvp": {"raw": 0.873, "norm": 0.35}, "stl_molformer": {"raw": 0.901, "norm": 0.65}}, {"group": "distal_safety", "dataset": "LD50_Zhu", "n": "7385", "metric": "mae", "mtl_dmpnn": {"raw": 0.69, "norm": 0.0}, "mtl_chemeleon": {"raw": 0.687, "norm": 0.0}, "mtl_graphmvp": {"raw": 0.708, "norm": 0.0}, "mtl_molformer": {"raw": 0.703, "norm": 0.0}, "stl_dmpnn": {"raw": 0.689, "norm": 0.0}, "stl_chemeleon": {"raw": 0.565, "norm": 0.95}, "stl_graphmvp": {"raw": 0.621, "norm": 0.73}, "stl_molformer": {"raw": 0.675, "norm": 0.18}}, {"group": "composite_exposure", "dataset": "Half_Life_Obach", "n": "667", "metric": "spearman", "mtl_dmpnn": {"raw": 0.525, "norm": 0.75}, "mtl_chemeleon": {"raw": 0.493, "norm": 0.7}, "mtl_graphmvp": {"raw": 0.418, "norm": 0.6}, "mtl_molformer": {"raw": 0.479, "norm": 0.65}, "stl_dmpnn": {"raw": 0.373, "norm": 0.55}, "stl_chemeleon": {"raw": 0.51, "norm": 0.75}, "stl_graphmvp": {"raw": 0.383, "norm": 0.55}, "stl_molformer": {"raw": 0.464, "norm": 0.65}}], "t3": [{"group": "oral_uptake", "dataset": "Caco2_Wang", "metric": "mae", "mtl_map": {"raw": 0.444, "norm": 0.29}, "mtl_mean": {"raw": 0.384, "norm": 0.38}, "mtl_max": {"raw": 0.359, "norm": 0.38}, "stl_map": {"raw": 0.371, "norm": 0.38}, "stl_mean": {"raw": 0.438, "norm": 0.29}, "stl_max": {"raw": 0.42, "norm": 0.29}}, {"group": "oral_uptake", "dataset": "HIA_Hou", "metric": "auroc", "mtl_map": {"raw": 0.987, "norm": 0.8}, "mtl_mean": {"raw": 0.966, "norm": 0.4}, "mtl_max": {"raw": 0.988, "norm": 0.8}, "stl_map": {"raw": 0.97, "norm": 0.4}, "stl_mean": {"raw": 0.963, "norm": 0.35}, "stl_max": {"raw": 0.977, "norm": 0.55}}, {"group": "oral_uptake", "dataset": "Solubility_AqSolDB", "metric": "mae", "mtl_map": {"raw": 0.931, "norm": 0.39}, "mtl_mean": {"raw": 0.95, "norm": 0.28}, "mtl_max": {"raw": 0.916, "norm": 0.39}, "stl_map": {"raw": 0.844, "norm": 0.44}, "stl_mean": {"raw": 0.864, "norm": 0.44}, "stl_max": {"raw": 0.828, "norm": 0.5}}, {"group": "distribution_cns", "dataset": "BBB_Martins", "metric": "auroc", "mtl_map": {"raw": 0.911, "norm": 0.77}, "mtl_mean": {"raw": 0.9, "norm": 0.54}, "mtl_max": {"raw": 0.909, "norm": 0.73}, "stl_map": {"raw": 0.893, "norm": 0.5}, "stl_mean": {"raw": 0.878, "norm": 0.38}, "stl_max": {"raw": 0.892, "norm": 0.5}}, {"group": "distribution_cns", "dataset": "Lipophilicity_AstraZeneca", "metric": "mae", "mtl_map": {"raw": 0.616, "norm": 0.29}, "mtl_mean": {"raw": 0.607, "norm": 0.29}, "mtl_max": {"raw": 0.58, "norm": 0.29}, "stl_map": {"raw": 0.481, "norm": 0.81}, "stl_mean": {"raw": 0.501, "norm": 0.81}, "stl_max": {"raw": 0.47, "norm": 0.9}}, {"group": "distribution_cns", "dataset": "PPBR_AZ", "metric": "mae", "mtl_map": {"raw": 7.833, "norm": 0.75}, "mtl_mean": {"raw": 7.711, "norm": 0.8}, "mtl_max": {"raw": 7.535, "norm": 0.9}, "stl_map": {"raw": 7.796, "norm": 0.75}, "stl_mean": {"raw": 8.011, "norm": 0.65}, "stl_max": {"raw": 7.834, "norm": 0.75}}, {"group": "distribution_cns", "dataset": "VDss_Lombardo", "metric": "spearman", "mtl_map": {"raw": 0.576, "norm": 0.68}, "mtl_mean": {"raw": 0.587, "norm": 0.74}, "mtl_max": {"raw": 0.606, "norm": 0.74}, "stl_map": {"raw": 0.681, "norm": 0.89}, "stl_mean": {"raw": 0.626, "norm": 0.79}, "stl_max": {"raw": 0.669, "norm": 0.89}}, {"group": "interaction_liabilities", "dataset": "CYP2C9_Veith", "metric": "auprc", "mtl_map": {"raw": 0.76, "norm": 0.55}, "mtl_mean": {"raw": 0.755, "norm": 0.5}, "mtl_max": {"raw": 0.769, "norm": 0.6}, "stl_map": {"raw": 0.774, "norm": 0.6}, "stl_mean": {"raw": 0.756, "norm": 0.5}, "stl_max": {"raw": 0.766, "norm": 0.55}}, {"group": "interaction_liabilities", "dataset": "CYP2D6_Veith", "metric": "auprc", "mtl_map": {"raw": 0.682, "norm": 0.63}, "mtl_mean": {"raw": 0.66, "norm": 0.53}, "mtl_max": {"raw": 0.681, "norm": 0.63}, "stl_map": {"raw": 0.671, "norm": 0.58}, "stl_mean": {"raw": 0.64, "norm": 0.37}, "stl_max": {"raw": 0.642, "norm": 0.37}}, {"group": "interaction_liabilities", "dataset": "CYP3A4_Veith", "metric": "auprc", "mtl_map": {"raw": 0.857, "norm": 0.47}, "mtl_mean": {"raw": 0.844, "norm": 0.32}, "mtl_max": {"raw": 0.868, "norm": 0.58}, "stl_map": {"raw": 0.862, "norm": 0.47}, "stl_mean": {"raw": 0.853, "norm": 0.42}, "stl_max": {"raw": 0.87, "norm": 0.58}}, {"group": "interaction_liabilities", "dataset": "Pgp_Broccatelli", "metric": "auroc", "mtl_map": {"raw": 0.925, "norm": 0.69}, "mtl_mean": {"raw": 0.917, "norm": 0.56}, "mtl_max": {"raw": 0.923, "norm": 0.62}, "stl_map": {"raw": 0.905, "norm": 0.5}, "stl_mean": {"raw": 0.907, "norm": 0.5}, "stl_max": {"raw": 0.937, "norm": 0.94}}, {"group": "interaction_liabilities", "dataset": "hERG", "metric": "auroc", "mtl_map": {"raw": 0.866, "norm": 0.8}, "mtl_mean": {"raw": 0.843, "norm": 0.7}, "mtl_max": {"raw": 0.883, "norm": 1.0}, "stl_map": {"raw": 0.825, "norm": 0.5}, "stl_mean": {"raw": 0.812, "norm": 0.5}, "stl_max": {"raw": 0.865, "norm": 0.8}}, {"group": "metabolic_turnover", "dataset": "CYP2C9_Substrate_CarbonMangels", "metric": "auprc", "mtl_map": {"raw": 0.453, "norm": 0.95}, "mtl_mean": {"raw": 0.435, "norm": 0.8}, "mtl_max": {"raw": 0.457, "norm": 0.95}, "stl_map": {"raw": 0.413, "norm": 0.6}, "stl_mean": {"raw": 0.399, "norm": 0.55}, "stl_max": {"raw": 0.427, "norm": 0.75}}, {"group": "metabolic_turnover", "dataset": "CYP2D6_Substrate_CarbonMangels", "metric": "auprc", "mtl_map": {"raw": 0.671, "norm": 0.39}, "mtl_mean": {"raw": 0.683, "norm": 0.5}, "mtl_max": {"raw": 0.705, "norm": 0.78}, "stl_map": {"raw": 0.649, "norm": 0.39}, "stl_mean": {"raw": 0.635, "norm": 0.39}, "stl_max": {"raw": 0.685, "norm": 0.5}}, {"group": "metabolic_turnover", "dataset": "CYP3A4_Substrate_CarbonMangels", "metric": "auroc", "mtl_map": {"raw": 0.668, "norm": 1.0}, "mtl_mean": {"raw": 0.673, "norm": 1.0}, "mtl_max": {"raw": 0.689, "norm": 1.0}, "stl_map": {"raw": 0.638, "norm": 0.58}, "stl_mean": {"raw": 0.612, "norm": 0.37}, "stl_max": {"raw": 0.634, "norm": 0.58}}, {"group": "metabolic_turnover", "dataset": "Clearance_Hepatocyte_AZ", "metric": "spearman", "mtl_map": {"raw": 0.561, "norm": 1.0}, "mtl_mean": {"raw": 0.497, "norm": 0.89}, "mtl_max": {"raw": 0.556, "norm": 1.0}, "stl_map": {"raw": 0.502, "norm": 0.94}, "stl_mean": {"raw": 0.466, "norm": 0.83}, "stl_max": {"raw": 0.517, "norm": 0.94}}, {"group": "metabolic_turnover", "dataset": "Clearance_Microsome_AZ", "metric": "spearman", "mtl_map": {"raw": 0.614, "norm": 0.75}, "mtl_mean": {"raw": 0.611, "norm": 0.75}, "mtl_max": {"raw": 0.616, "norm": 0.75}, "stl_map": {"raw": 0.599, "norm": 0.75}, "stl_mean": {"raw": 0.572, "norm": 0.45}, "stl_max": {"raw": 0.605, "norm": 0.75}}, {"group": "distal_safety", "dataset": "AMES", "metric": "auroc", "mtl_map": {"raw": 0.842, "norm": 0.45}, "mtl_mean": {"raw": 0.831, "norm": 0.4}, "mtl_max": {"raw": 0.848, "norm": 0.65}, "stl_map": {"raw": 0.833, "norm": 0.4}, "stl_mean": {"raw": 0.821, "norm": 0.3}, "stl_max": {"raw": 0.84, "norm": 0.45}}, {"group": "distal_safety", "dataset": "DILI", "metric": "auroc", "mtl_map": {"raw": 0.91, "norm": 0.75}, "mtl_mean": {"raw": 0.902, "norm": 0.65}, "mtl_max": {"raw": 0.905, "norm": 0.65}, "stl_map": {"raw": 0.899, "norm": 0.6}, "stl_mean": {"raw": 0.892, "norm": 0.6}, "stl_max": {"raw": 0.902, "norm": 0.65}}, {"group": "distal_safety", "dataset": "LD50_Zhu", "metric": "mae", "mtl_map": {"raw": 0.703, "norm": 0.0}, "mtl_mean": {"raw": 0.699, "norm": 0.0}, "mtl_max": {"raw": 0.687, "norm": 0.0}, "stl_map": {"raw": 0.559, "norm": 0.95}, "stl_mean": {"raw": 0.62, "norm": 0.73}, "stl_max": {"raw": 0.565, "norm": 0.95}}, {"group": "composite_exposure", "dataset": "Half_Life_Obach", "metric": "spearman", "mtl_map": {"raw": 0.471, "norm": 0.65}, "mtl_mean": {"raw": 0.463, "norm": 0.65}, "mtl_max": {"raw": 0.493, "norm": 0.7}, "stl_map": {"raw": 0.511, "norm": 0.75}, "stl_mean": {"raw": 0.452, "norm": 0.65}, "stl_max": {"raw": 0.51, "norm": 0.75}}], "t4": [{"task": "HIA_Hou", "metric": "auroc", "within_score": 0.99177, "within_norm": 0.95, "within_rank": 2, "cross_score": 0.993416, "cross_norm": 1, "cross_rank": 1}, {"task": "Pgp_Broccatelli", "metric": "auroc", "within_score": 0.935191, "within_norm": 0.9375, "within_rank": 2, "cross_score": 0.939549, "cross_norm": 1, "cross_rank": 1}, {"task": "Bioavailability_Ma", "metric": "auroc", "within_score": 0.708946, "within_norm": 0, "within_rank": 21, "cross_score": 0.709012, "cross_norm": 0, "cross_rank": 21}, {"task": "BBB_Martins", "metric": "auroc", "within_score": 0.912871, "within_norm": 0.807692, "within_rank": 6, "cross_score": 0.915416, "cross_norm": 0.884615, "cross_rank": 4}, {"task": "CYP2C9_Veith", "metric": "auprc", "within_score": 0.785247, "within_norm": 0.7, "within_rank": 7, "cross_score": 0.789991, "cross_norm": 0.8, "cross_rank": 5}, {"task": "CYP2D6_Veith", "metric": "auprc", "within_score": 0.713775, "within_norm": 0.684211, "within_rank": 7, "cross_score": 0.722866, "cross_norm": 0.842105, "cross_rank": 4}, {"task": "CYP3A4_Veith", "metric": "auprc", "within_score": 0.877553, "within_norm": 0.736842, "within_rank": 6, "cross_score": 0.881571, "cross_norm": 0.789474, "cross_rank": 5}, {"task": "CYP2C9_Substrate_CarbonMangels", "metric": "auprc", "within_score": 0.447329, "within_norm": 0.95, "within_rank": 2, "cross_score": 0.439888, "cross_norm": 0.9, "cross_rank": 3}, {"task": "CYP2D6_Substrate_CarbonMangels", "metric": "auprc", "within_score": 0.698162, "within_norm": 0.666667, "within_rank": 7, "cross_score": 0.696022, "cross_norm": 0.666667, "cross_rank": 7}, {"task": "CYP3A4_Substrate_CarbonMangels", "metric": "auroc", "within_score": 0.657414, "within_norm": 0.842105, "within_rank": 4, "cross_score": 0.664331, "cross_norm": 0.947368, "cross_rank": 2}, {"task": "hERG", "metric": "auroc", "within_score": 0.877526, "within_norm": 0.95, "within_rank": 2, "cross_score": 0.882769, "cross_norm": 1, "cross_rank": 1}, {"task": "AMES", "metric": "auroc", "within_score": 0.855833, "within_norm": 0.8, "within_rank": 5, "cross_score": 0.860865, "cross_norm": 0.8, "cross_rank": 5}, {"task": "DILI", "metric": "auroc", "within_score": 0.934, "within_norm": 0.95, "within_rank": 2, "cross_score": 0.937826, "cross_norm": 0.95, "cross_rank": 2}, {"task": "Caco2_Wang", "metric": "mae", "within_score": 0.307912, "within_norm": 0.666667, "within_rank": 9, "cross_score": 0.298224, "cross_norm": 0.666667, "cross_rank": 9}, {"task": "Lipophilicity_AstraZeneca", "metric": "mae", "within_score": 0.451664, "within_norm": 1, "within_rank": 1, "cross_score": 0.445075, "cross_norm": 1, "cross_rank": 1}, {"task": "Solubility_AqSolDB", "metric": "mae", "within_score": 0.774777, "within_norm": 0.888889, "within_rank": 3, "cross_score": 0.760583, "cross_norm": 0.944444, "cross_rank": 2}, {"task": "PPBR_AZ", "metric": "mae", "within_score": 7.034619, "within_norm": 1, "within_rank": 1, "cross_score": 6.942401, "cross_norm": 1, "cross_rank": 1}, {"task": "VDss_Lombardo", "metric": "spearman", "within_score": 0.686908, "within_norm": 0.894737, "within_rank": 3, "cross_score": 0.692326, "cross_norm": 0.894737, "cross_rank": 3}, {"task": "Half_Life_Obach", "metric": "spearman", "within_score": 0.550741, "within_norm": 0.85, "within_rank": 4, "cross_score": 0.559823, "cross_norm": 0.9, "cross_rank": 3}, {"task": "Clearance_Hepatocyte_AZ", "metric": "spearman", "within_score": 0.495275, "within_norm": 0.888889, "within_rank": 3, "cross_score": 0.503428, "cross_norm": 0.944444, "cross_rank": 2}, {"task": "Clearance_Microsome_AZ", "metric": "spearman", "within_score": 0.629052, "within_norm": 0.95, "within_rank": 2, "cross_score": 0.635766, "cross_norm": 1, "cross_rank": 1}, {"task": "LD50_Zhu", "metric": "mae", "within_score": 0.588429, "within_norm": 0.818182, "within_rank": 5, "cross_score": 0.582675, "cross_norm": 0.909091, "cross_rank": 3}], "t2": [{"group": "Oral uptake", "tasks": "Solubility_AqSolDB; Caco2_Wang; HIA_Hou; Bioavailability_Ma", "rationale": "Solubility to intestinal absorption and systemic availability"}, {"group": "Distribution / CNS", "tasks": "Lipophilicity_AstraZeneca; PPBR_AZ; BBB_Martins; VDss_Lombardo", "rationale": "Physicochemical distribution and CNS exposure"}, {"group": "Interaction liabilities", "tasks": "Pgp_Broccatelli; hERG; CYP(2C9/2D6/3A4)_Veith", "rationale": "Transporter, ion-channel, and enzyme interaction risks"}, {"group": "Metabolic turnover", "tasks": "CYP(2C9/2D6/3A4)_Substrate_CarboMangels; Clearance_(Hepatocyte/Microsome)_AZ", "rationale": "Substrate recognition and hepatic elimination"}, {"group": "Distal safety", "tasks": "AMES; DILI; LD50_Zhu", "rationale": "Mutagenicity, organ toxicity, and acute toxicity"}, {"group": "Composite exposure", "tasks": "Half_Life_Obach", "rationale": "Composite endpoint isolated to reduce gradient conflict"}]}""")
_ST = json.loads(r"""{"deployable_single": {"CheMeleon·STL|within": {"median": 0.19999999999999996, "mean": 0.2086848095238095, "W": 225.0, "p": 7.046295299214032e-05, "wtl": "19/0/2", "base_mean": 0.6452380952380953}, "CheMeleon·STL|cross": {"median": 0.25, "mean": 0.25188628571428573, "W": 229.0, "p": 3.965369774338429e-05, "wtl": "20/0/1", "base_mean": 0.6452380952380953}, "MAP·STL|within": {"median": 0.266842, "mean": 0.24535147619047623, "W": 223.0, "p": 9.311011342738716e-05, "wtl": "19/0/2", "base_mean": 0.6085714285714285}, "MAP·STL|cross": {"median": 0.286667, "mean": 0.28855295238095247, "W": 228.0, "p": 4.5990462883872256e-05, "wtl": "20/0/1", "base_mean": 0.6085714285714285}, "D-MPNN·MTL|within": {"median": 0.26769199999999993, "mean": 0.24249433333333337, "W": 169.0, "p": 0.0001382081160141556, "wtl": "17/3/1", "base_mean": 0.6114285714285714}, "D-MPNN·MTL|cross": {"median": 0.29000000000000004, "mean": 0.2856958095238096, "W": 187.0, "p": 0.00010667725000605988, "wtl": "18/2/1", "base_mean": 0.6114285714285714}}, "per_task_oracle": {"within": {"median": 0.004736999999999991, "mean": 0.029161, "W": 107.0, "p": 0.17440805990734143, "wtl": "11/3/7", "base_mean": 0.8247619047619046}, "cross": {"median": 0.050000000000000044, "mean": 0.07236247619047619, "W": 139.5, "p": 0.009329438406863406, "wtl": "13/3/5", "base_mean": 0.8247619047619046}}, "n_tasks": 21, "ensemble_mean": {"within": 0.8539229047619046, "cross": 0.8971243809523809}}""")
print("setup ok | tasks:", len(_PT["t1"]), "| final ensemble:", _GT["final_ensemble"])


In [ ]:
# === figure1.png (within-seed / matched-seed track basis) ===
import json, numpy as np, matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.patches import Patch
apply_figure_style(sizes=(8,7,6))

_FIG1 = json.loads(r"""{"solo": {"stl_chemeleon": 0.6331340234535723, "mtl_map": 0.6172810307208804, "mtl_dmpnn": 0.6046659306997653, "stl_map": 0.604456675321337, "mtl_chemeleon": 0.5997610020166411, "mtl_molformer": 0.5540811736300458, "mtl_graphmvp": 0.5244573582919447, "stl_molformer": 0.4626633669866752, "stl_graphmvp": 0.4160323455436237, "stl_dmpnn": 0.20975553964275767}, "pertask": [{"task": "Solubility_AqSolDB", "mtl_dmpnn": 0.6111111111111112, "stl_dmpnn": 0.0555555555555555, "stl_map": 0.4444444444444444, "stl_chemeleon": 0.5, "stl_graphmvp": 0.4444444444444444, "stl_molformer": 0.4444444444444444, "mtl_graphmvp": 0.2777777777777778, "mtl_molformer": 0.3888888888888889, "mtl_chemeleon": 0.2777777777777778, "mtl_map": 0.3888888888888889}, {"task": "Caco2_Wang", "mtl_dmpnn": 0.625, "stl_dmpnn": 0.25, "stl_map": 0.375, "stl_chemeleon": 0.25, "stl_graphmvp": 0.2916666666666667, "stl_molformer": 0.2916666666666667, "mtl_graphmvp": 0.375, "mtl_molformer": 0.3333333333333333, "mtl_chemeleon": 0.375, "mtl_map": 0.2916666666666667}, {"task": "HIA_Hou", "mtl_dmpnn": 0.95, "stl_dmpnn": 0.1, "stl_map": 0.4, "stl_chemeleon": 0.35, "stl_graphmvp": 0.35, "stl_molformer": 0.55, "mtl_graphmvp": 0.2, "mtl_molformer": 0.6, "mtl_chemeleon": 0.8, "mtl_map": 0.8}, {"task": "Lipophilicity_AstraZeneca", "mtl_dmpnn": 0.7142857142857143, "stl_dmpnn": 0.2857142857142857, "stl_map": 0.8095238095238095, "stl_chemeleon": 0.8571428571428571, "stl_graphmvp": 0.9047619047619048, "stl_molformer": 0.3809523809523809, "mtl_graphmvp": 0.2857142857142857, "mtl_molformer": 0.1428571428571428, "mtl_chemeleon": 0.2857142857142857, "mtl_map": 0.2857142857142857}, {"task": "PPBR_AZ", "mtl_dmpnn": 1.0, "stl_dmpnn": 0.15, "stl_map": 0.8, "stl_chemeleon": 0.75, "stl_graphmvp": 0.7, "stl_molformer": 0.6, "mtl_graphmvp": 0.95, "mtl_molformer": 0.65, "mtl_chemeleon": 0.9, "mtl_map": 0.7}, {"task": "BBB_Martins", "mtl_dmpnn": 0.5, "stl_dmpnn": 0.0, "stl_map": 0.4615384615384615, "stl_chemeleon": 0.4615384615384615, "stl_graphmvp": 0.3846153846153846, "stl_molformer": 0.3076923076923077, "mtl_graphmvp": 0.3846153846153846, "mtl_molformer": 0.5384615384615384, "mtl_chemeleon": 0.6923076923076923, "mtl_map": 0.7307692307692307}, {"task": "VDss_Lombardo", "mtl_dmpnn": 0.7894736842105263, "stl_dmpnn": 0.6842105263157895, "stl_map": 0.8947368421052632, "stl_chemeleon": 0.8947368421052632, "stl_graphmvp": 0.7368421052631579, "stl_molformer": 0.7368421052631579, "mtl_graphmvp": 0.6842105263157895, "mtl_molformer": 0.6842105263157895, "mtl_chemeleon": 0.7368421052631579, "mtl_map": 0.6842105263157895}, {"task": "Pgp_Broccatelli", "mtl_dmpnn": 0.5625, "stl_dmpnn": 0.125, "stl_map": 0.5, "stl_chemeleon": 0.9375, "stl_graphmvp": 0.25, "stl_molformer": 0.5, "mtl_graphmvp": 0.625, "mtl_molformer": 0.5, "mtl_chemeleon": 0.625, "mtl_map": 0.6875}, {"task": "hERG", "mtl_dmpnn": 0.75, "stl_dmpnn": 0.35, "stl_map": 0.5, "stl_chemeleon": 0.8, "stl_graphmvp": 0.35, "stl_molformer": 0.5, "mtl_graphmvp": 0.5, "mtl_molformer": 0.9, "mtl_chemeleon": 0.7, "mtl_map": 0.8}, {"task": "CYP2C9_Veith", "mtl_dmpnn": 0.3, "stl_dmpnn": 0.2, "stl_map": 0.6, "stl_chemeleon": 0.55, "stl_graphmvp": 0.35, "stl_molformer": 0.5, "mtl_graphmvp": 0.3, "mtl_molformer": 0.5, "mtl_chemeleon": 0.6, "mtl_map": 0.55}, {"task": "CYP2D6_Veith", "mtl_dmpnn": 0.5789473684210527, "stl_dmpnn": 0.2105263157894736, "stl_map": 0.5789473684210527, "stl_chemeleon": 0.3684210526315789, "stl_graphmvp": 0.3684210526315789, "stl_molformer": 0.3684210526315789, "mtl_graphmvp": 0.4736842105263157, "mtl_molformer": 0.5263157894736842, "mtl_chemeleon": 0.631578947368421, "mtl_map": 0.631578947368421}, {"task": "CYP3A4_Veith", "mtl_dmpnn": 0.2105263157894736, "stl_dmpnn": 0.1578947368421052, "stl_map": 0.4736842105263157, "stl_chemeleon": 0.5789473684210527, "stl_graphmvp": 0.4210526315789473, "stl_molformer": 0.2631578947368421, "mtl_graphmvp": 0.2631578947368421, "mtl_molformer": 0.2631578947368421, "mtl_chemeleon": 0.5789473684210527, "mtl_map": 0.4736842105263157}, {"task": "CYP2C9_Substrate_CarbonMangels", "mtl_dmpnn": 1.0, "stl_dmpnn": 0.15, "stl_map": 0.6, "stl_chemeleon": 0.75, "stl_graphmvp": 0.35, "stl_molformer": 0.5, "mtl_graphmvp": 0.95, "mtl_molformer": 0.75, "mtl_chemeleon": 0.75, "mtl_map": 0.95}, {"task": "CYP2D6_Substrate_CarbonMangels", "mtl_dmpnn": 0.3888888888888889, "stl_dmpnn": 0.1666666666666666, "stl_map": 0.3888888888888889, "stl_chemeleon": 0.6111111111111112, "stl_graphmvp": 0.1666666666666666, "stl_molformer": 0.3888888888888889, "mtl_graphmvp": 0.6111111111111112, "mtl_molformer": 0.6666666666666666, "mtl_chemeleon": 0.3888888888888889, "mtl_map": 0.4444444444444444}, {"task": "CYP3A4_Substrate_CarbonMangels", "mtl_dmpnn": 0.7894736842105263, "stl_dmpnn": 0.0526315789473684, "stl_map": 0.5789473684210527, "stl_chemeleon": 0.5263157894736842, "stl_graphmvp": 0.2631578947368421, "stl_molformer": 0.2631578947368421, "mtl_graphmvp": 1.0, "mtl_molformer": 0.9473684210526316, "mtl_chemeleon": 0.9473684210526316, "mtl_map": 1.0}, {"task": "Clearance_Hepatocyte_AZ", "mtl_dmpnn": 0.2777777777777778, "stl_dmpnn": 0.1666666666666666, "stl_map": 0.8333333333333334, "stl_chemeleon": 0.5555555555555556, "stl_graphmvp": 0.2777777777777778, "stl_molformer": 0.8888888888888888, "mtl_graphmvp": 0.8333333333333334, "mtl_molformer": 0.9444444444444444, "mtl_chemeleon": 0.5555555555555556, "mtl_map": 0.9444444444444444}, {"task": "Clearance_Microsome_AZ", "mtl_dmpnn": 0.65, "stl_dmpnn": 0.15, "stl_map": 0.75, "stl_chemeleon": 0.75, "stl_graphmvp": 0.3, "stl_molformer": 0.45, "mtl_graphmvp": 0.75, "mtl_molformer": 0.75, "mtl_chemeleon": 0.75, "mtl_map": 0.75}, {"task": "AMES", "mtl_dmpnn": 0.3, "stl_dmpnn": 0.2, "stl_map": 0.4, "stl_chemeleon": 0.45, "stl_graphmvp": 0.2, "stl_molformer": 0.3, "mtl_graphmvp": 0.3, "mtl_molformer": 0.3, "mtl_chemeleon": 0.65, "mtl_map": 0.45}, {"task": "DILI", "mtl_dmpnn": 0.95, "stl_dmpnn": 0.4, "stl_map": 0.6, "stl_chemeleon": 0.65, "stl_graphmvp": 0.35, "stl_molformer": 0.65, "mtl_graphmvp": 0.65, "mtl_molformer": 0.6, "mtl_chemeleon": 0.65, "mtl_map": 0.75}, {"task": "LD50_Zhu", "mtl_dmpnn": 0.0, "stl_dmpnn": 0.0, "stl_map": 0.9545454545454546, "stl_chemeleon": 0.9545454545454546, "stl_graphmvp": 0.7272727272727273, "stl_molformer": 0.1818181818181818, "mtl_graphmvp": 0.0, "mtl_molformer": 0.0, "mtl_chemeleon": 0.0, "mtl_map": 0.0}, {"task": "Half_Life_Obach", "mtl_dmpnn": 0.75, "stl_dmpnn": 0.55, "stl_map": 0.75, "stl_chemeleon": 0.75, "stl_graphmvp": 0.55, "stl_molformer": 0.65, "mtl_graphmvp": 0.6, "mtl_molformer": 0.65, "mtl_chemeleon": 0.7, "mtl_map": 0.65}], "groups": {"Oral uptake": ["Solubility_AqSolDB", "Caco2_Wang", "HIA_Hou"], "Distribution/CNS": ["Lipophilicity_AstraZeneca", "PPBR_AZ", "BBB_Martins", "VDss_Lombardo"], "Interaction": ["Pgp_Broccatelli", "hERG", "CYP2C9_Veith", "CYP2D6_Veith", "CYP3A4_Veith"], "Metabolism": ["CYP2C9_Substrate_CarbonMangels", "CYP2D6_Substrate_CarbonMangels", "CYP3A4_Substrate_CarbonMangels", "Clearance_Hepatocyte_AZ", "Clearance_Microsome_AZ"], "Safety": ["AMES", "DILI", "LD50_Zhu"], "Exposure": ["Half_Life_Obach"]}}""")

PRETTY = {"stl_chemeleon":"CheMeleon\u00b7STL","mtl_chemeleon":"CheMeleon\u00b7MTL",
 "stl_map":"MAP\u00b7STL","mtl_map":"MAP\u00b7MTL","stl_dmpnn":"D-MPNN\u00b7STL","mtl_dmpnn":"D-MPNN\u00b7MTL",
 "stl_graphmvp":"GraphMVP\u00b7STL","mtl_graphmvp":"GraphMVP\u00b7MTL","stl_molformer":"MoLFormer\u00b7STL","mtl_molformer":"MoLFormer\u00b7MTL"}
SHORT = {"Solubility_AqSolDB":"Solubility","Caco2_Wang":"Caco2","HIA_Hou":"HIA","Lipophilicity_AstraZeneca":"Lipophilicity_AZ",
 "PPBR_AZ":"PPBR_AZ","BBB_Martins":"BBB","VDss_Lombardo":"VDss","Pgp_Broccatelli":"Pgp","hERG":"hERG",
 "CYP2C9_Veith":"CYP2C9","CYP2D6_Veith":"CYP2D6","CYP3A4_Veith":"CYP3A4","CYP2C9_Substrate_CarbonMangels":"CYP2C9_Substrate",
 "CYP2D6_Substrate_CarbonMangels":"CYP2D6_Substrate","CYP3A4_Substrate_CarbonMangels":"CYP3A4_Substrate",
 "Clearance_Hepatocyte_AZ":"Clearance_Hepatocyte_AZ","Clearance_Microsome_AZ":"Clearance_Microsome_AZ",
 "AMES":"AMES","DILI":"DILI","LD50_Zhu":"LD50","Half_Life_Obach":"Half_Life"}
FINAL5 = {"mtl_dmpnn","mtl_molformer","stl_chemeleon","stl_graphmvp","stl_map"}
ENS,OTHER,STLc,MTLc = "#c0392b","#b7b7b7","#4c72b0","#dd8452"
GROUPS = _FIG1["groups"]

solo = dict(sorted(_FIG1["solo"].items(), key=lambda kv:-kv[1]))
task_order = [t for g in GROUPS.values() for t in g]
bb = ["dmpnn","chemeleon","graphmvp","molformer"]
pt = {e["task"]:e for e in _FIG1["pertask"]}
stl_by_task = [[pt[t]["stl_"+b] for b in bb] for t in task_order]
mtl_by_task = [[pt[t]["mtl_"+b] for b in bb] for t in task_order]
stl_all = np.array([v for r in stl_by_task for v in r]); mtl_all = np.array([v for r in mtl_by_task for v in r])

fig = plt.figure(figsize=(13,5.2))
gs = fig.add_gridspec(1,2,width_ratios=[1,1.5],wspace=0.28)
axA = fig.add_subplot(gs[0,0]); axB = fig.add_subplot(gs[0,1])

order = list(solo.keys()); vals=[solo[m] for m in order]
ybars = np.arange(len(order))[::-1]
colors = [ENS if m in FINAL5 else OTHER for m in order]
axA.barh(ybars, vals, color=colors, edgecolor="white", height=0.72)
axA.set_yticks(ybars); axA.set_yticklabels([PRETTY[m] for m in order])
for y,v in zip(ybars,vals): axA.text(v+0.008,y,f"{v:.3f}",va="center",ha="left",fontsize=7)
axA.set_xlim(0,0.75)
axA.set_xlabel("mean matched-seed track test norm_score\n(21 tasks, 1 = leaderboard SOTA)")
axA.set_title("a  No single backbone dominates",loc="left",fontweight="bold",fontsize=9)
axA.legend(handles=[Patch(color=ENS,label="selected into final k=5 ensemble"),Patch(color=OTHER,label="not selected")],loc="lower right",fontsize=6.5,frameon=False)

n=len(task_order); xpos=np.arange(n); w=0.36
bpS=axB.boxplot(stl_by_task,positions=xpos-w/2,widths=w*0.9,patch_artist=True,showfliers=False,medianprops=dict(color="black",lw=1))
bpM=axB.boxplot(mtl_by_task,positions=xpos+w/2,widths=w*0.9,patch_artist=True,showfliers=False,medianprops=dict(color="black",lw=1))
for b in bpS["boxes"]: b.set(facecolor=STLc,alpha=0.55,edgecolor=STLc)
for b in bpM["boxes"]: b.set(facecolor=MTLc,alpha=0.55,edgecolor=MTLc)
for bp,cc in [(bpS,STLc),(bpM,MTLc)]:
    for wk in bp["whiskers"]: wk.set(color=cc)
    for cap in bp["caps"]: cap.set(color=cc)
rng=np.random.default_rng(0)
for i,(s,mm_) in enumerate(zip(stl_by_task,mtl_by_task)):
    axB.scatter((i-w/2)+rng.uniform(-0.06,0.06,len(s)),s,s=8,color=STLc,zorder=3,edgecolor="white",linewidth=0.3)
    axB.scatter((i+w/2)+rng.uniform(-0.06,0.06,len(mm_)),mm_,s=8,color=MTLc,zorder=3,edgecolor="white",linewidth=0.3)
axB.set_xticks(xpos); axB.set_xticklabels([SHORT[t] for t in task_order],rotation=55,ha="right",fontsize=5.5)
axB.set_ylabel("matched-seed track norm_score  (1 = SOTA)"); axB.set_ylim(-0.03,1.08)
axB.set_title("b  MTL vs STL: neither wins every task",loc="left",fontweight="bold",fontsize=9)
axB.legend(handles=[Patch(color=STLc,alpha=0.6,label=f"STL  (pooled \u03bc={stl_all.mean():.3f})"),Patch(color=MTLc,alpha=0.6,label=f"MTL  (pooled \u03bc={mtl_all.mean():.3f})")],loc="lower left",fontsize=6.5,frameon=False)
acc=0
for gname,tasks in GROUPS.items():
    st,en=acc,acc+len(tasks); acc=en
    if st>0: axB.axvline(st-0.5,color="grey",ls=":",lw=0.6,alpha=0.6)
    axB.text((st+en-1)/2,1.05,gname,ha="center",va="bottom",fontsize=6.5,style="italic",color="#444")
fig.savefig(os.path.join(OUTDIR,"figure1.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# === figure2.png ===
import json, numpy as np, matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.patches import Patch

META_GREY = "#888888"




GT = _GT
PT = _PT
apply_figure_style(sizes=(8,7,6))

STLc="#4c72b0"; MTLc="#dd8452"

t3=PT["t3"]
tasks3=[e["dataset"] for e in t3]
cols3=["mtl_map","mtl_mean","mtl_max","stl_map","stl_mean","stl_max"]
collab=["MAP","mean","max","MAP","mean","max"]
M=np.array([[e[c]["norm"] for c in cols3] for e in t3])  # tasks x 6

d_mean_stl=[e["stl_map"]["norm"]-e["stl_mean"]["norm"] for e in t3]
d_mean_mtl=[e["mtl_map"]["norm"]-e["mtl_mean"]["norm"] for e in t3]
d_max_stl=[e["stl_map"]["norm"]-e["stl_max"]["norm"] for e in t3]
d_max_mtl=[e["mtl_map"]["norm"]-e["mtl_max"]["norm"] for e in t3]

sw_mean_stl=sum(1 for x in d_mean_stl if x>1e-9); sw_mean_mtl=sum(1 for x in d_mean_mtl if x>1e-9)
sw_max_stl=sum(1 for x in d_max_stl if x>1e-9); sw_max_mtl=sum(1 for x in d_max_mtl if x>1e-9)

plt.close('all')
fig=plt.figure(figsize=(15.5,7.6))
gs=fig.add_gridspec(1,3, width_ratios=[1.2,1.05,1.05], wspace=0.5)
axH=fig.add_subplot(gs[0,0])
im=axH.imshow(M, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
axH.set_xticks(range(6)); axH.set_xticklabels(collab, fontsize=6.5)
axH.set_yticks(range(len(tasks3))); axH.set_yticklabels(tasks3, fontsize=5.5)
axH.axvline(2.5, color="black", lw=1.2)
axH.text(1.0, -1.2, "MTL", ha="center", fontweight="bold", fontsize=8)
axH.text(4.0, -1.2, "STL", ha="center", fontweight="bold", fontsize=8)
for i in range(len(tasks3)):
    for j in range(6):
        v=M[i,j]
        axH.text(j,i,f"{v:.2f}", ha="center", va="center", fontsize=4.6,
                 color="black" if 0.25<v<0.85 else "white")
axH.set_title("a  Aggregation per task", loc="left", fontweight="bold", fontsize=9, pad=20)
cbar=fig.colorbar(im, ax=axH, fraction=0.04, pad=0.09, orientation="horizontal", location="bottom")
cbar.set_label("norm_score (1 = SOTA)", fontsize=6.5); cbar.ax.tick_params(labelsize=5.5)

def delta_panel(ax, ds, dm, title, subtitle):
    y=np.arange(len(tasks3))
    ax.barh(y-0.2, ds, height=0.38, color=STLc, label="STL", alpha=0.85)
    ax.barh(y+0.2, dm, height=0.38, color=MTLc, label="MTL", alpha=0.85)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_yticks(y); ax.set_yticklabels(tasks3, fontsize=5.2)
    ax.invert_yaxis()
    ax.set_xlabel("Δ norm_score  (MAP − aggregation)", fontsize=7)
    ax.set_title(title, loc="left", fontweight="bold", fontsize=9, pad=16)
    ax.text(0.0,1.012, subtitle, transform=ax.transAxes, ha="left", fontsize=6.2, color="#444")
    ax.legend(fontsize=6.5, frameon=False, loc="lower right")

axB=fig.add_subplot(gs[0,1]); axC=fig.add_subplot(gs[0,2])
delta_panel(axB, d_mean_stl, d_mean_mtl, "b  MAP vs 3-backbone mean",
            f"MAP wins {sw_mean_stl}/21 STL, {sw_mean_mtl}/21 MTL — raises the floor")
delta_panel(axC, d_max_stl, d_max_mtl, "c  MAP vs task-best single backbone",
            f"MAP wins {sw_max_stl}/21 STL, {sw_max_mtl}/21 MTL — dilutes best signal")
fig.savefig(os.path.join(OUTDIR,"figure2.png"), dpi=300, bbox_inches="tight")

In [ ]:
# === figure3.png ===
import json
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# skill:figure-style kernel.py (auto-injected on skill load)
META_GREY = "#888888"




apply_figure_style(sizes=(8, 7, 6))

GT = _GT

ks = GT["ksweep"]
kk = [r[0] for r in ks]
w_top = [r[1] for r in ks]; w_marg = [r[2] for r in ks]
c_top = [r[3] for r in ks]; c_marg = [r[4] for r in ks]

plt.close('all')
fig = plt.figure(figsize=(13, 5))
gs = fig.add_gridspec(1, 2, width_ratios=[1.15, 1], wspace=0.26)
axL = fig.add_subplot(gs[0, 0]); axR = fig.add_subplot(gs[0, 1])
axL.plot(kk, w_top, "-o", color="#4c72b0", label="matched-seed track", markersize=5, lw=1.6)
axL.plot(kk, c_top, "-s", color="#dd8452", label="seed-averaged track", markersize=5, lw=1.6)
axL.axvline(5, color="#c0392b", ls="--", lw=1.2)
axL.text(5.15, 0.56, "selected k = 5", color="#c0392b", fontsize=7.5, rotation=90, va="bottom")
axL.set_xlabel("ensemble size  k  (number of base models)")
axL.set_ylabel("validation norm_score\n(mean of top-20 combos)")
axL.set_xticks(kk)
axL.set_title("a  Validation performance saturates at k=5", loc="left", fontweight="bold", fontsize=9)
axL.legend(fontsize=7, frameon=False, loc="lower right"); axL.margins(x=0.03)

# Panel B sign-based, separate within & cross bars
xb = np.array(kk[1:])
gw = w_marg[1:]; gc = c_marg[1:]
bw = 0.38
def sc(g): return "#c0392b" if g >= 0 else "#5aa06f"
axR.bar(xb - bw/2, gw, width=bw, color=[sc(g) for g in gw], edgecolor="white")
axR.bar(xb + bw/2, gc, width=bw, color=[sc(g) for g in gc], edgecolor="white", hatch="////")
axR.axhline(0, color="black", lw=0.8)
axR.axvline(5.5, color="#888", ls=":", lw=1)
axR.text(6.7, 0.14, "plateau →\ngains ≈ 0, then < 0", fontsize=6.8, color="#555", ha="center")
axR.set_xlabel("k  (marginal gain of adding the k-th model)")
axR.set_ylabel("Δ validation norm_score / model")
axR.set_xticks(xb)
axR.set_title("b  Diminishing returns beyond k=5", loc="left", fontweight="bold", fontsize=9)
axR.legend(handles=[Patch(facecolor="#c0392b", label="matched-seed track, gain ≥ 0"),
                    Patch(facecolor="#c0392b", hatch="////", label="seed-averaged track, gain ≥ 0"),
                    Patch(facecolor="#5aa06f", label="gain < 0 (saturated)")],
           fontsize=6, frameon=False, loc="upper right")
fig.savefig(os.path.join(OUTDIR,"figure3.png"), dpi=300, bbox_inches="tight")
print("saved figure3.png")

In [ ]:
# === figure4.png ===
import json, numpy as np, matplotlib as mpl, matplotlib.pyplot as plt
import base64, io, pandas as pd
from scipy.stats import spearmanr, kendalltau
from matplotlib.patches import Patch

META_GREY = "#888888"




apply_figure_style(sizes=(8,7,6))

GT = _GT
PT = _PT

PRETTY = {
 "stl_chemeleon":"CheMeleon·STL","mtl_chemeleon":"CheMeleon·MTL",
 "stl_map":"MAP·STL","mtl_map":"MAP·MTL",
 "stl_dmpnn":"D-MPNN·STL","mtl_dmpnn":"D-MPNN·MTL",
 "stl_graphmvp":"GraphMVP·STL","mtl_graphmvp":"GraphMVP·MTL",
 "stl_molformer":"MoLFormer·STL","mtl_molformer":"MoLFormer·MTL",
}
FINAL = set(GT["final_ensemble"])

STLc="#4c72b0"; MTLc="#dd8452"

vt=pd.read_csv(CSV_PATH)

rw,_=spearmanr(vt.val_within, vt.test_within)
rc,_=spearmanr(vt.val_cross, vt.test_cross)
val_comb=pd.concat([vt.val_within, vt.val_cross]); test_comb=pd.concat([vt.test_within, vt.test_cross])
rcomb2,_=spearmanr(vt.val_comb, vt.test_comb)
tw,_=kendalltau(vt.val_within,vt.test_within); tc,_=kendalltau(vt.val_cross,vt.test_cross); tcm,_=kendalltau(vt.val_comb,vt.test_comb)

FINAL_KEY=",".join(sorted(GT["final_ensemble"]))

ls=GT["loo_shapley"]
members=["mtl_dmpnn","mtl_molformer","stl_chemeleon","stl_map","stl_graphmvp"]
mpretty=[PRETTY[m] for m in members]

plt.close('all')
fig=plt.figure(figsize=(13.5,9.5))
gs=fig.add_gridspec(2,2, hspace=0.40, wspace=0.28)

# Panel A: scatter val_within vs test_within, val_cross vs test_cross
axA=fig.add_subplot(gs[0,0])
axA.scatter(vt.val_within, vt.test_within, s=16, color="#4c72b0", alpha=0.55, edgecolor="none", label=f"matched-seed track  (ρ={rw:.3f})")
axA.scatter(vt.val_cross, vt.test_cross, s=16, color="#dd8452", alpha=0.55, edgecolor="none", label=f"seed-averaged track  (ρ={rc:.3f})")
fk=vt[vt.combo_key==FINAL_KEY].iloc[0]
axA.scatter([fk.val_within],[fk.test_within], s=90, marker="*", color="#c0392b", edgecolor="black", lw=0.5, zorder=5)
axA.scatter([fk.val_cross],[fk.test_cross], s=90, marker="*", color="#c0392b", edgecolor="black", lw=0.5, zorder=5, label="final k=5 ensemble")
axA.set_xlabel("validation norm_score"); axA.set_ylabel("held-out test norm_score")
axA.set_title("a  Validation reliably ranks test performance", loc="left", fontweight="bold", fontsize=9)
axA.legend(fontsize=6.5, frameon=False, loc="lower right")
axA.text(0.02,0.97, f"n = 133 fusion configs · all p < 1e-24\ncombined ρ={rcomb2:.3f}",
         transform=axA.transAxes, va="top", fontsize=6.3, color="#444")

# Panel B: LOO delta-norm per member (within & cross).
axB=fig.add_subplot(gs[0,1])
loo_w=[ls[m][1] for m in members]
loo_c=[ls[m][3] for m in members]
y=np.arange(len(members))[::-1]
bw=0.38
axB.barh(y+bw/2, loo_w, height=bw, color="#4c72b0", label="matched-seed track", alpha=0.85)
axB.barh(y-bw/2, loo_c, height=bw, color="#dd8452", label="seed-averaged track", alpha=0.85)
axB.axvline(0, color="black", lw=0.8)
axB.set_yticks(y); axB.set_yticklabels(mpretty, fontsize=7)
axB.set_xlabel("norm_score drop when member removed  (LOO)")
axB.set_title("b  Leave-one-out: unequal, complementary roles", loc="left", fontweight="bold", fontsize=9)
axB.legend(fontsize=6.5, frameon=False, loc="center right")
axB.text(0.98,0.06, "GraphMVP·STL: removal slightly\nimproves consensus (< 0)", transform=axB.transAxes,
         ha="right", fontsize=6, color="#5aa06f")

# Panel C: Shapley contributions per member (within & cross)
axC=fig.add_subplot(gs[1,0])
sh_w=[ls[m][4] for m in members]; sh_c=[ls[m][5] for m in members]
axC.barh(y+bw/2, sh_w, height=bw, color="#4c72b0", label="matched-seed track", alpha=0.85)
axC.barh(y-bw/2, sh_c, height=bw, color="#dd8452", label="seed-averaged track", alpha=0.85)
axC.set_yticks(y); axC.set_yticklabels(mpretty, fontsize=7)
axC.set_xlabel("Shapley contribution to ensemble norm_score")
axC.set_title("c  Shapley: all members contribute (globally)", loc="left", fontweight="bold", fontsize=9)
axC.legend(fontsize=6.5, frameon=False, loc="lower right")
for yi,(w_,c_) in zip(y,zip(sh_w,sh_c)):
    axC.text(max(w_,c_)+0.004, yi, f"{w_:.3f}/{c_:.3f}", va="center", fontsize=5.5, color="#444")
axC.set_xlim(0, 0.27)

# Panel D: task-wise winner distribution (cross-seed, tasks) + correlation callout
axD=fig.add_subplot(gs[1,1])
win=GT["winners"]["cross"]
wmembers=["mtl_dmpnn","mtl_molformer","stl_map","stl_chemeleon","stl_graphmvp"]
wvals=[win[m] for m in wmembers]
wlab=[PRETTY[m] for m in wmembers]
yb=np.arange(len(wmembers))[::-1]
bars=axD.barh(yb, wvals, color="#6a51a3", alpha=0.85, edgecolor="white")
axD.set_yticks(yb); axD.set_yticklabels(wlab, fontsize=7)
for y_,v in zip(yb,wvals):
    axD.text(v+0.1, y_, str(v), va="center", fontsize=7)
axD.set_xlabel("# tasks where model is the best single base  (seed-averaged track, 21 tasks)")
axD.set_title("d  Gain comes from task-wise specialization", loc="left", fontweight="bold", fontsize=9)
axD.set_xlim(0,7.2)
corr=GT["corr"]["cross"]
axD.text(0.98,0.30,
         f"Yet member predictions are highly correlated:\n  pred corr = {corr['pred_offdiag']:.3f}   residual corr = {corr['resid_offdiag']:.3f}\n→ gain is NOT error decorrelation",
         transform=axD.transAxes, ha="right", va="top", fontsize=6.2, color="#c0392b",
         bbox=dict(boxstyle="round,pad=0.4", fc="#fdf2f0", ec="#c0392b", lw=0.6))
axD.text(0.98,0.06, f"best single (MAP·STL) mean rank {GT['best_single']['cross']['mean_rank']:.2f}\n→ ensemble mean rank {GT['best_single']['cross']['ens_rank']:.2f}",
         transform=axD.transAxes, ha="right", va="bottom", fontsize=6, color="#333")

# Remove floating callout; add compact note near GraphMVP negative bars
axB.texts[-1].remove()
gy = np.arange(len(members))[::-1][members.index("stl_graphmvp")]
axB.annotate("removal improves\nconsensus (Δ<0)", xy=(-0.006, gy), xytext=(-0.013, gy+0.9),
             fontsize=5.8, color="#3a7d44", ha="left", va="center")

fig.savefig(os.path.join(OUTDIR,"figure4.png"), dpi=300, bbox_inches="tight")

In [ ]:
# === figure5.png ===
import json
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import wilcoxon

META_GREY = "#888888"




apply_figure_style(sizes=(8, 7, 6))

GT = _GT
PT = _PT

t1 = {e["dataset"]: e for e in PT["t1"]}
t4 = {e["task"]: e for e in PT["t4"] if e["task"] != "Bioavailability_Ma"}
t3 = {e["dataset"]: e for e in PT["t3"]}

models = ["mtl_dmpnn", "mtl_chemeleon", "mtl_graphmvp", "mtl_molformer",
          "stl_dmpnn", "stl_chemeleon", "stl_graphmvp", "stl_molformer"]

tasks = [d for d in t4 if d in t1]
rows = []
for d in tasks:
    ens_w = t4[d]["within_norm"]
    ens_c = t4[d]["cross_norm"]
    singles_w = [t1[d][m]["norm"] for m in models] + [t3[d]["mtl_map"]["norm"], t3[d]["stl_map"]["norm"]]
    best_w = max(x for x in singles_w if x is not None)
    rows.append((d, ens_w, ens_c, best_w))

ens_w = np.array([r[1] for r in rows])
ens_c = np.array([r[2] for r in rows])
best = np.array([r[3] for r in rows])


def per_task_norm(model):
    if model in ("stl_map", "mtl_map"):
        return np.array([t3[d][model]["norm"] for d in tasks])
    return np.array([t1[d][model]["norm"] for d in tasks])


results = {}
for base_name, base_model in [("CheMeleon·STL", "stl_chemeleon"), ("MAP·STL", "stl_map"), ("D-MPNN·MTL", "mtl_dmpnn")]:
    b = per_task_norm(base_model)
    for track, ens in [("within", ens_w), ("cross", ens_c)]:
        diff = ens - b
        stat, p = wilcoxon(ens, b, alternative="greater")
        wins = int((diff > 1e-9).sum())
        ties = int((abs(diff) <= 1e-9).sum())
        losses = int((diff < -1e-9).sum())
        results[(base_name, track)] = dict(median=float(np.median(diff)), mean=float(diff.mean()),
                                           W=float(stat), p=float(p),
                                           wtl=f"{wins}/{ties}/{losses}", base_mean=float(b.mean()))

oracle = {}
for track, ens in [("within", ens_w), ("cross", ens_c)]:
    diff = ens - best
    stat, p = wilcoxon(ens, best, alternative="greater")
    wins = int((diff > 1e-9).sum())
    ties = int((abs(diff) <= 1e-9).sum())
    losses = int((diff < -1e-9).sum())
    oracle[track] = dict(median=float(np.median(diff)), mean=float(diff.mean()),
                         W=float(stat), p=float(p),
                         wtl=f"{wins}/{ties}/{losses}", base_mean=float(best.mean()))

plt.close('all')

fig = plt.figure(figsize=(13, 5.2))
gs = fig.add_gridspec(1, 2, width_ratios=[1.25, 1], wspace=0.3)
axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])

base_ch = per_task_norm("stl_chemeleon")
xj = [0, 1]
for i in range(len(tasks)):
    axA.plot(xj, [base_ch[i], ens_c[i]], '-', color="#bbbbbb", lw=0.6, alpha=0.7, zorder=1)
axA.scatter([0] * len(tasks), base_ch, s=22, color="#4c72b0", zorder=3, label="CheMeleon·STL (best single)")
axA.scatter([1] * len(tasks), ens_c, s=22, color="#c0392b", zorder=3, label="k=5 ensemble (seed-averaged track)")
axA.set_xticks(xj)
axA.set_xticklabels(["best deployable\nsingle model", "k=5 late-fusion\nensemble"])
axA.set_xlim(-0.35, 1.35)
axA.set_ylabel("per-task norm_score (1 = SOTA)")
axA.set_title("a  Ensemble lifts nearly every task", loc="left", fontweight="bold", fontsize=9)
r = results[("CheMeleon·STL", "cross")]
axA.text(0.5, 0.04, f"paired Wilcoxon: p = {r['p']:.1e}\nwins/ties/losses = {r['wtl']}\nmedian Δ = {r['median']:+.3f}",
         transform=axA.transAxes, ha="center", fontsize=7, color="#333",
         bbox=dict(boxstyle="round,pad=0.4", fc="#f7f7f7", ec="#999", lw=0.5))
axA.legend(fontsize=6.8, frameon=False, loc="upper left")

labels = []
meds = []
ps = []
cols = []
order = [("CheMeleon·STL", "within"), ("CheMeleon·STL", "cross"),
         ("MAP·STL", "within"), ("MAP·STL", "cross"),
         ("D-MPNN·MTL", "within"), ("D-MPNN·MTL", "cross"),
         ("per-task oracle", "within"), ("per-task oracle", "cross")]
for base, track in order:
    if base == "per-task oracle":
        v = oracle[track]
    else:
        v = results[(base, track)]
    labels.append(f"vs {base} · {'matched-seed track' if track=='within' else 'seed-averaged track'}")
    meds.append(v["mean"])
    ps.append(v["p"])
    cols.append("#c0392b" if track == "cross" else "#4c72b0")
y = np.arange(len(labels))[::-1]
axB.scatter(meds, y, s=45, color=cols, zorder=3)
axB.hlines(y, 0, meds, color=cols, lw=1.2, alpha=0.5)
axB.axvline(0, color="black", lw=0.8)
for yi, (m_, p_) in zip(y, zip(meds, ps)):
    star = "***" if p_ < 0.001 else "**" if p_ < 0.01 else "*" if p_ < 0.05 else "n.s."
    axB.text(m_ + 0.006, yi, f"{m_:+.3f} {star}", va="center", fontsize=6.3)
axB.set_yticks(y)
axB.set_yticklabels(labels, fontsize=6.5)
axB.set_xlabel("mean Δ norm_score  (ensemble − baseline)")
axB.set_xlim(-0.02, 0.34)
axB.set_title("b  Gain is significant vs every\n    deployable single model", loc="left", fontweight="bold", fontsize=9)
axB.text(0.98, 0.02, "*** Holm p<0.001   * p<0.05   n.s. not sig.\noracle = per-task best single (upper bound)",
         transform=axB.transAxes, ha="right", fontsize=5.6, color="#666")
fig.savefig(os.path.join(OUTDIR,"figure5.png"), dpi=300, bbox_inches="tight")
print("saved figure5.png")